In [55]:
# -----------------------------
# 1. Importaciones Librerías
# -----------------------------

In [56]:
# ------------------------Librerias------------------------------------
from google.cloud import bigquery 

import os
import gdown
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import gdown
import os
import gdown
import os

In [57]:
# -----------------------------
# 2. Funciones 
# -----------------------------

In [58]:
# Dataframe Metadata sitios 
def separar_direccion(df, columna='address'):
   
    resultados = {
        'street_address': [],
        'city': [],
        'zip_code': []
    }

    for row in df[columna]:
        try:
            parts = [p.strip() for p in row.split(',')]

            street_address = parts[1]
            city = parts[2]

            state_zip = parts[3].split()
            zip_code = state_zip[1] if len(state_zip) > 1 else None

            resultados['street_address'].append(street_address)
            resultados['city'].append(city)
            resultados['zip_code'].append(zip_code)

        except Exception as e:
            print(f"Error procesando fila: {row} -> {e}")
            resultados['street_address'].append(None)
            resultados['city'].append(None)
            resultados['zip_code'].append(None)

    df = df.copy()
    for key, values in resultados.items():
        df[key] = values

    return df

In [59]:
# -----------------------------
# 3. Carga / procesamiento de datos
# -----------------------------

In [60]:
## -----------------------------------------------------Review-sitios--------------------------------------------------------


url_folder_sitios = 'https://drive.google.com/drive/folders/1olnuKLjT8W2QnCUUwh8uDuTTKVZyxQ0Z'  # metadata-sitios
output_folder = os.path.join("maps", "metadata-sitios")

# Descargar la carpeta
#gdown.download_folder(url_folder_sitios, output=output_folder, quiet=False, use_cookies=False)
archivos_json = glob.glob(os.path.join(r'maps\metadata-sitios', "*.json"))

# Se guardará archivos JSON en formato pickle (binario) el cual es más rápido para futuras consultas
for archivo in archivos_json:
    df = pd.read_json(archivo, lines=True)
    df.to_pickle(archivo.replace(".json", ".pkl"))  # Guardar como .pkl

In [61]:
#gdown.download_folder(url_folder_sitios,output=r'Maps\metadata-sitios',quiet=False, use_cookies=False)
# %%
# Carga los archivos Pickle generados y los combina en un solo DataFrame.
df_list = [pd.read_pickle(archivo.replace(".json", ".pkl")) for archivo in archivos_json] # convierte el nombre de cada archivo de .json a .pkl para cargarlo correctamente.
metadata_sitio = pd.concat(df_list, ignore_index=True)

# %%
metadata_sitio = metadata_sitio.drop(columns=['price','state', 'hours', 'relative_results', 'url'])

# reviso que categorias hay. Busco los restaurantes de pizza
# df_mt_sitios['category'].unique()
unique_categories = pd.DataFrame(metadata_sitio["category"].explode().unique() )
#filtra por categoria restaurante
unique_resto = unique_categories[unique_categories[0].str.lower().str.contains("restaurant", na=False)]

# selecciono las filas que tienen pizza restaurant
df_mt_sitios_pizza = metadata_sitio[metadata_sitio['category'].apply(lambda x: isinstance(x, list) and 'Pizza restaurant' in x)]

# Extract state (two uppercase letters) using regex
df_mt_sitios_pizza['state'] = df_mt_sitios_pizza['address'].str.extract(r',\s*([A-Z]{2})\s*\d{5}') 
# selecciono las pizzerias de CA y NV

#filtra por estado y numero de reviews>50
df_mt_pizza_CANV = df_mt_sitios_pizza[df_mt_sitios_pizza['state'].isin(['CA', 'NV'])]
df_mt_pizza_CANV=df_mt_pizza_CANV[df_mt_pizza_CANV['num_of_reviews'] > 100] # filtro por numero de reviews

df_mt_pizza_CANV=separar_direccion(df_mt_pizza_CANV)

# Drop the original column if necessary
df_mt_pizza_CANV.drop(columns=['description', 'category','address'], inplace=True)

# renombro y reordeno las columnas
df_mt_pizza_CANV.rename(columns={'name': 'business_name', 'street_address': 'address'}, inplace=True)

#cols = df_mt_pizza_CANV.columns.tolist()
cols = ['gmap_id', 'business_name', 'address' , 'city', 'state', 'zip_code',  'latitude', 'longitude', 'avg_rating', 'num_of_reviews','MISC']
df_mt_pizza_CANV = df_mt_pizza_CANV[cols] 

# elimino los duplicados manteniendo la primera instancia
df_mt_pizza_CANV_sindup = df_mt_pizza_CANV.drop_duplicates(subset=['gmap_id'], keep='first')



In [62]:
# -----------------------------
# Procesamiento de etiquetas MISC
# guardamos los atributos mas comunes , 3 por categoria, lo hacemos con con  review_count > 100
import ast
import pandas as pd
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer

# Lista fija de etiquetas permitidas
etiquetas_destacadas_fijas = [
    'Delivery', 'Takeout', 'Dine-in',
    'Staff required to disinfect surfaces between visits', 'Temperature check required',
    'Reservations required', 'Dinner', 'Lunch', 'Solo dining',
    'Wheelchair accessible entrance', 'Wheelchair accessible parking lot', 'Wheelchair accessible restroom',
    'Comfort food', 'Quick bite', 'Vegetarian options', 'Dessert',
    'Good for kids', 'High chairs', 'Restroom',
    'Casual', 'Cozy', 'Cosy', 'Groups', 'Tourists', 'College students',
    'Accepts reservations', 'Usually a wait', 'Dinner reservations recommended',
    'NFC mobile payments', 'Debit cards', 'Credit cards',
    'Fast service', 'Sports', 'LGBTQ friendly',
    'Identifies as women-led', 'Identifies as veteran-led'
]

# Filtrar etiquetas válidas
def extraer_etiquetas_fijas(misc):
    etiquetas = []
    if isinstance(misc, dict):
        for lista in misc.values():
            if isinstance(lista, list):
                etiquetas.extend(lista)
    # Retornar solo las válidas
    return [et for et in etiquetas if et in etiquetas_destacadas_fijas]

# --- 1. Limpiar y preparar campo MISC ---
df_mt_pizza_CANV_sindup = df_mt_pizza_CANV_sindup[df_mt_pizza_CANV_sindup['MISC'].notna()]
df_mt_pizza_CANV_sindup['MISC'] = df_mt_pizza_CANV_sindup['MISC'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# --- 2. Extraer etiquetas permitidas ---
df_mt_pizza_CANV_sindup['etiquetas_seleccionadas'] = df_mt_pizza_CANV_sindup['MISC'].apply(extraer_etiquetas_fijas)

# --- 3. Crear dummies de forma segura ---
mlb = MultiLabelBinarizer(classes=etiquetas_destacadas_fijas)
df_dummies = pd.DataFrame(
    mlb.fit_transform(df_mt_pizza_CANV_sindup['etiquetas_seleccionadas']),
    columns=mlb.classes_,
    index=df_mt_pizza_CANV_sindup.index
)

# --- 4. Concatenar ---
df_mt_pizza_CANV_sindup.drop(columns=['etiquetas_seleccionadas'], inplace=True)
df_mt_pizza_CANV_sindup = pd.concat([df_mt_pizza_CANV_sindup, df_dummies], axis=1)



In [63]:
df_mt_pizza_CANV_sindup.drop(columns=["MISC"],inplace=True)
df_mt_pizza_CANV_sindup["city"] =df_mt_pizza_CANV_sindup["city"].str.lower()

In [ ]:
# Verificacion- coordenadas

import time
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

# Inicializar el geolocalizador
geolocator = Nominatim(user_agent="geo_checker")

# Función con sleep y manejo de errores
def get_lat_long_safe(address, city, state, zip_code):
    full_address = f"{address}, {city}, {state} {zip_code}"
    try:
        location = geolocator.geocode(full_address, timeout=10)
        #time.sleep(1)  # espera de 1 segundo para evitar bloqueos
        if location:
            return location.latitude, location.longitude
        else:
            return None, None
    except GeocoderTimedOut:
        return None, None
    except Exception as e:
        print(f"Error geocoding '{full_address}': {e}")
        return None, None

from time import sleep

# Lista para guardar coordenadas
verified_coords = []

# Recorremos las filas del DataFrame
for i, row in df_mt_pizza_CANV_sindup.iterrows():
    # Extraemos los datos necesarios
    address = row['address']
    city = row['city']
    state = row['state']
    zip_code = row['zip_code']

    # Llamamos a tu función de geocodificación
    lat, lon = get_lat_long_safe(address, city, state, zip_code)

    # Agregamos a la lista
    verified_coords.append((lat, lon))

    # Si estás usando Nominatim, agregar pausa para evitar bloqueo
    #sleep(1)

# Convertimos la lista de tuplas a DataFrame
verified_df = pd.DataFrame(verified_coords, columns=["verified_latitude", "verified_longitude"])

# Lo agregamos al DataFrame original
df_mt_pizza_CANV_sindup = pd.concat([df_mt_pizza_CANV_sindup.reset_index(drop=True), verified_df], axis=1)

# Compare the existing vs verified coordinates
df_mt_pizza_CANV_sindup['lat_diff'] = abs(df_mt_pizza_CANV_sindup['latitude'] - df_mt_pizza_CANV_sindup['verified_latitude'])
df_mt_pizza_CANV_sindup['long_diff'] = abs(df_mt_pizza_CANV_sindup['longitude'] - df_mt_pizza_CANV_sindup['verified_longitude'])

# Check rows where the difference is significant (e.g., > 0.01 degrees)
incorrect_coords = df_mt_pizza_CANV_sindup[(df_mt_pizza_CANV_sindup['lat_diff'] > 0.01) | (df_mt_pizza_CANV_sindup['long_diff'] > 0.01)]

df_mt_pizza_CANV_sindup['lat_diff'] = abs(df_mt_pizza_CANV_sindup['latitude'] - df_mt_pizza_CANV_sindup['verified_latitude'])

# reemplazo las latitudes por las correctas
df_mt_pizza_CANV_sindup['latitude'] = df_mt_pizza_CANV_sindup.apply(lambda x: x['verified_latitude'] if x['lat_diff'] > 0.01 else x['latitude'], axis=1)

# reemplazo las longitudes por las correctas
df_mt_pizza_CANV_sindup['longitude'] = df_mt_pizza_CANV_sindup.apply(lambda x: x['verified_longitude'] if x['long_diff'] > 0.01 else x['longitude'], axis=1)

# elimino las columnas intermedias
df_mt_pizza_CANV_sindup.drop(columns=['verified_latitude', 'verified_longitude', 'lat_diff', 'long_diff'], inplace=True)

# guardo df_mt_pizza_CANV final 202502081554
df_mt_pizza_CANV_sindup.to_parquet(r'archivos_ETL_intermedios\google_pizza_final.parquet' , engine='fastparquet')

In [ ]:
## --------------------------------------------------Reviews-estados--------------------------------------------------------------------------

# Crear carpetas si no existen
os.makedirs(r"Maps\review-Nevada", exist_ok=True)
os.makedirs(r"Maps\review-California", exist_ok=True)

# Convertir todos los JSON a PKL (si aún no lo hiciste)
def convertir_json_a_pkl(carpeta):
    archivos_json = glob.glob(os.path.join(carpeta, "*.json"))
    for archivo in archivos_json:
        df = pd.read_json(archivo, lines=True)
        archivo_pkl = archivo.replace(".json", ".pkl")
        df.to_pickle(archivo_pkl)
        print(f"✅ Guardado: {archivo_pkl}")

# Ejecutar conversión en ambas carpetas
convertir_json_a_pkl(r"Maps\review-California")
convertir_json_a_pkl(r"Maps\review-Nevada")

print("\n✅ Conversión a .pkl completa.")

# Unir todos los .pkl sin agregar columna 'estado'
def cargar_pickles(carpeta):
    archivos_pkl = glob.glob(os.path.join(carpeta, "*.pkl"))
    dfs = [pd.read_pickle(archivo) for archivo in archivos_pkl]
    return pd.concat(dfs, ignore_index=True)

# Cargar los datos
df_california = cargar_pickles(r"Maps\review-California")
df_nevada = cargar_pickles(r"Maps\review-Nevada")

# Unir los DataFrames
df_rev_CA_NV = pd.concat([df_california, df_nevada], ignore_index=True)

# Eliminamos, por ahora, pics y resp. no usaremos esas columnas
df_rev_CA = df_rev_CA_NV.drop(columns=['pics', 'resp'])
# # reviso si hay reviews duplicadas
df_dup = df_rev_CA_NV[df_rev_CA_NV.duplicated(subset=['user_id', 'gmap_id', 'time'], keep=False)]
df_dup = df_dup.sort_values(['user_id', 'gmap_id', 'time'])

# eliminar los duplicados manteniendo la primera instancia
df_rev_sindup = df_rev_CA.drop_duplicates(subset=['user_id', 'gmap_id', 'time'], keep='first')
# rename column name to user_name
df_rev_sindup.rename(columns={'name': 'user_name'}, inplace=True)

# convertir columna time en int64 a datetime
df_rev_sindup['date'] = pd.to_datetime(df_rev_sindup['time'], unit='ms')
# Eliminamos la columan time
df_rev_CA_sindup = df_rev_sindup.drop(columns=['time'])


# guardo df_rev_NJ_sindup
df_rev_CA_sindup.to_parquet(r'archivos_ETL_intermedios\google_review_final.parquet' , engine='fastparquet')

✅ Guardado: Maps\review-California\1.pkl
✅ Guardado: Maps\review-California\10.pkl
✅ Guardado: Maps\review-California\11.pkl
✅ Guardado: Maps\review-California\12.pkl
✅ Guardado: Maps\review-California\13.pkl
✅ Guardado: Maps\review-California\14.pkl
✅ Guardado: Maps\review-California\15.pkl
✅ Guardado: Maps\review-California\16.pkl
✅ Guardado: Maps\review-California\17.pkl
✅ Guardado: Maps\review-California\18.pkl
✅ Guardado: Maps\review-California\2.pkl
✅ Guardado: Maps\review-California\3.pkl
✅ Guardado: Maps\review-California\4.pkl
✅ Guardado: Maps\review-California\5.pkl
✅ Guardado: Maps\review-California\6.pkl
✅ Guardado: Maps\review-California\7.pkl
✅ Guardado: Maps\review-California\8.pkl
✅ Guardado: Maps\review-California\9.pkl
✅ Guardado: Maps\review-Nevada\1.pkl
✅ Guardado: Maps\review-Nevada\10.pkl
✅ Guardado: Maps\review-Nevada\11.pkl
✅ Guardado: Maps\review-Nevada\12.pkl
✅ Guardado: Maps\review-Nevada\2.pkl
✅ Guardado: Maps\review-Nevada\3.pkl
✅ Guardado: Maps\review-Neva

In [ ]:
## ---------------------------------------------------YELP - Business-----------------------------------------------------------------------------------


url_file_yelp_business = 'https://drive.google.com/uc?id=1byFtzpZXopdCN-XYmMHMpZqzgAqfQBBu'
#gdown.download(url_file_yelp_business, output=r'Yelp\business.pkl', quiet=False, use_cookies=False)

business = pd.read_pickle(r'Yelp\business.pkl')

# Paso 1: Identificar columnas duplicadas por nombre
cols = business.columns
duplicated_cols = cols[cols.duplicated()].unique()

# Paso 2: Crear un nuevo DataFrame combinando los datos de las columnas duplicadas
for col in duplicated_cols:
    # Seleccionamos todas las columnas con ese nombre (ej: todas las 'name')
    same_cols = business.loc[:, business.columns == col]
    
    # Combinamos: si la primera tiene NaN, usa el valor de la otra
    business[col] = same_cols.bfill(axis=1).iloc[:, 0]

# Paso 3: Eliminar las columnas duplicadas (con el mismo nombre repetido)
business = business.loc[:, ~business.columns.duplicated()]


# Eliminamos columnas que no serán utilizadas
business = business.drop(columns=["is_open", "attributes",  "hours"])

business = business[business['state'].notna()]

# Guardar como Parquet (más eficiente) y obtener su tamaño
#business.to_parquet(r"C:\Users\felip\Desktop\Stuff\Cursos\SoyHenry\Clases\LABS\PF\PF_google_yelp\datasets\business.parquet")
#print("Peso del Parquet:", os.path.getsize(r"C:\Users\felip\Desktop\Stuff\Cursos\SoyHenry\Clases\LABS\PF\PF_google_yelp\datasets\business.parquet") / (1024 * 1024), "MB")

business.to_parquet(r'archivos_ETL_intermedios\business.parquet' , engine='fastparquet')

# Filtra por categorias restaurant y pizza
unique_categories = pd.DataFrame(business['categories'].unique())

unique_resto = unique_categories[unique_categories[0].str.lower().str.contains("restaurant", na=False)]

unique_pizza = unique_categories[unique_categories[0].str.lower().str.contains("pizza", na=False)]

# selecciono los locales que venden pizza. Para los estados de CA y NV.
# Remember: The & and | operators require both sides of the condition to be in parentheses when used in pandas filtering.
# Your best bet is to use .isin(), as it is more readable and efficient:
yelp_pizza_CANV = business[
    business['categories'].str.lower().str.contains("pizza", na=False) &
    business['state'].isin(['CA', 'NV'])
]
yelp_pizza_CANV['state'].unique()

# reviso duplicados
df_dup = yelp_pizza_CANV[yelp_pizza_CANV.duplicated(subset=['business_id'], keep=False)]
df_dup = df_dup.sort_values(['business_id'])
# pasamos a minusculas las ciudades
yelp_pizza_CANV["city"] = yelp_pizza_CANV["city"].str.lower()
yelp_pizza_CANV["city"] = yelp_pizza_CANV["city"].str.strip()
# Eliminamos columnas que no serán utilizadas
yelp_pizza_CANV = yelp_pizza_CANV.drop(columns=["categories"])
yelp_pizza_CANV= isin=yelp_pizza_CANV[yelp_pizza_CANV["review_count"]>10]




In [ ]:
# ----------------------------- aplicar filtro carga incremental yelp

#yelp_pizza_CANV= isin=yelp_pizza_CANV[yelp_pizza_CANV["review_count"]>20]

In [ ]:
# guardo el archivo en formato parquet
yelp_pizza_CANV.to_parquet(r'archivos_ETL_intermedios\yelp_pizza_CANV.parquet' , engine='fastparquet')

In [ ]:

#----------------------------------- YELP-Review-----------------------------------------------------------------

url_file = "https://drive.google.com/uc?id=1mwNNdOMSNty6WumYdH9FJNJZJYQ6oD1c" # review.json 
#gdown.download(url_file, output="Yelp/review.json", quiet=False)

archivo = r'Yelp\review.json'
chunks = pd.read_json(archivo, lines=True, chunksize=100000)  # Leer en bloques de 100,000 filas

# Guardar cada chunk en un archivo pickle
# Esto evita que la memoria se llene al procesar el JSON por partes.
for i, chunk in enumerate(chunks):
    chunk.to_pickle(r'archivos_ETL_intermedios\review_part_{i}.pkl')  # Guarda en archivos más pequeños
    print(f'Chunk {i} guardado.')

archivos_pkl = glob.glob(r'archivos_ETL_intermedios\review_part_*.pkl')
# Leer y concatenar los DataFrames
df_reviews = pd.concat([pd.read_pickle(f) for f in archivos_pkl])

# Convertir la columna a tipo datetime
# Vamos a asegurarnos de que la columna "date" esté en el formato correcto:
df_reviews['date'] = pd.to_datetime(df_reviews['date'], errors='coerce')

# df_rev.to_parquet('/yelp/reviews.parquet' , engine='fastparquet')
# El motor pyarrow maneja mejor archivos grandes y evita problemas de compresión:
df_reviews.to_parquet(r'Archivos_ETL_intermedios\yelp_reviews_final.parquet', engine='pyarrow', compression='snappy')


Chunk 0 guardado.
Chunk 1 guardado.
Chunk 2 guardado.
Chunk 3 guardado.
Chunk 4 guardado.
Chunk 5 guardado.
Chunk 6 guardado.
Chunk 7 guardado.
Chunk 8 guardado.
Chunk 9 guardado.
Chunk 10 guardado.
Chunk 11 guardado.
Chunk 12 guardado.
Chunk 13 guardado.
Chunk 14 guardado.
Chunk 15 guardado.
Chunk 16 guardado.
Chunk 17 guardado.
Chunk 18 guardado.
Chunk 19 guardado.
Chunk 20 guardado.
Chunk 21 guardado.
Chunk 22 guardado.
Chunk 23 guardado.
Chunk 24 guardado.
Chunk 25 guardado.
Chunk 26 guardado.
Chunk 27 guardado.
Chunk 28 guardado.
Chunk 29 guardado.
Chunk 30 guardado.
Chunk 31 guardado.
Chunk 32 guardado.
Chunk 33 guardado.
Chunk 34 guardado.
Chunk 35 guardado.
Chunk 36 guardado.
Chunk 37 guardado.
Chunk 38 guardado.
Chunk 39 guardado.
Chunk 40 guardado.
Chunk 41 guardado.
Chunk 42 guardado.
Chunk 43 guardado.
Chunk 44 guardado.
Chunk 45 guardado.
Chunk 46 guardado.
Chunk 47 guardado.
Chunk 48 guardado.
Chunk 49 guardado.
Chunk 50 guardado.
Chunk 51 guardado.
Chunk 52 guardado.
Chu

In [ ]:
df_reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
6900000,BLu89xFjBFjcFB11ExqmbA,N3_CVCqI9bBVdrynd2KZVw,jh8j-DWqgWkbRe_a2XtKFQ,5,0,0,0,"Terrific place, I've known some of the staff f...",2020-12-17 02:02:53
6900001,0-JvMzZV1sy8mN5-b4mKng,nUbcXBO4tLHlm0SUHy9Qig,lhiaIa2-FkK1k5k-BeMxhg,5,3,0,0,My previous dentist died from a fast moving ca...,2018-07-06 01:14:47
6900002,yiBzu4GfyABEBImLR62uAw,_pNerxa8_oa2rGSsK8sckw,U1Gc3wKN5viu_e68KJVOdQ,5,3,1,2,"When it's 10PM on a Friday during COVID times,...",2021-01-09 17:03:54
6900003,wb3B-fwzK0JItr5spJMxLg,OIaACBkKofGy1QJw8hOYTg,_OxsSJLOrhdJq3-EewouvQ,5,0,0,0,Dane was outstanding and worked very hard to m...,2021-08-10 23:28:42
6900004,Qr_trHV78rMtTWJgTJhwEw,eWDVAz99_h2dSef6VF0Gzw,Bg7D8LrsW9XbYlEhT9yekw,1,0,0,0,Very small was looking for the Van Gogh exhibi...,2021-08-11 00:49:25


In [ ]:

# -----------------------------------------ciudades(Censo)-------------------------------------------------------------
import requests  # type: ignore
from bs4 import BeautifulSoup  # type: ignore
import pandas as pd  # type: ignore

# Estados a procesar
states = [ "california", "nevada"]

for state in states:
    url = f"https://www.{state}-demographics.com/cities_by_population"
    print(f"Procesando: {state.title()} - {url}")

    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    table = soup.find("table")
    if not table:
        print(f"⚠️ No se encontró una tabla en {url}")
        continue

    headers = [th.text.strip() for th in table.find_all("th")]
    data = []
    for row in table.find_all("tr")[1:]:
        cols = [td.text.strip() for td in row.find_all("td")]
        if cols:
            data.append(cols)

    df = pd.DataFrame(data, columns=headers)
    df.to_csv(f"archivos_ETL_intermedios/{state}_population.csv", index=False)
    print(f"✅ Guardado: {state}_population.csv\n")

Procesando: California - https://www.california-demographics.com/cities_by_population
✅ Guardado: california_population.csv

Procesando: Nevada - https://www.nevada-demographics.com/cities_by_population
✅ Guardado: nevada_population.csv



In [ ]:
# Cargar los CSVs
df_california = pd.read_csv("archivos_ETL_intermedios/california_population.csv")
df_nevada = pd.read_csv("archivos_ETL_intermedios/nevada_population.csv")

# Añadir columna 'Estado'
df_california["Estado"] = "Ca"
df_nevada["Estado"] = "Nv"

# Unir ambos
df_combinado = pd.concat([df_california, df_nevada], ignore_index=True)
df_combinado.drop(columns=["Rank"], inplace=True)
# Asegúrate del nombre exacto de la columna. En este ejemplo usamos "Population"
col_name = "Population"

# Eliminar filas donde no haya datos de población
df_combinado = df_combinado[df_combinado[col_name].notna()]

df_combinado.drop_duplicates(subset=["City"], inplace=True)


# Limpiar valores: quitar comas, convertir a número
df_combinado[col_name] = df_combinado[col_name].str.replace(",", "").str.strip()

# Eliminar posibles vacíos después de limpiar
df_combinado = df_combinado[df_combinado[col_name] != ""]

# Convertir a entero
df_combinado[col_name] = df_combinado[col_name].astype(int)

# renombrar la columna de ciudad a 'city'
df_combinado.rename(columns={"City": "city"}, inplace=True)

df_combinado['city'] =df_combinado['city'].str.lower()

# Ordenar por población descendente
df_combinado_sorted = df_combinado.sort_values(by=col_name, ascending=False)

# Generar ID incremental
df_combinado_sorted.reset_index(drop=True, inplace=True)

# Guardar en Parquet
df_combinado_sorted.to_parquet("archivos_ETL_intermedios/population_sorted.parquet", index=True)


print("✅ population_sorted.parquet generado y ordenado correctamente.")

✅ population_sorted.parquet generado y ordenado correctamente.


In [ ]:
# -----------------------------
# 3. Combinacion de tablas
# -----------------------------

In [ ]:
# ---------------------------------------------------- GOOGLE ---------------------------------------------------------------------
#primero filtramos por ciudad 
df_ciudades= pd.read_parquet(r'archivos_ETL_intermedios\population_sorted.parquet' , engine='fastparquet')
df_mt_pizza_CANV_sindup= pd.read_parquet(r'archivos_ETL_intermedios\google_pizza_final.parquet' , engine='fastparquet')

df_mt_pizza_CANV_sindup = df_mt_pizza_CANV_sindup[df_mt_pizza_CANV_sindup['city'].isin(df_ciudades['city'])]
df_mt_pizza_CANV_sindup.info()
#df_ciudades.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25 entries, 0 to 26
Data columns (total 46 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   gmap_id                                              25 non-null     object 
 1   business_name                                        25 non-null     object 
 2   address                                              25 non-null     object 
 3   city                                                 25 non-null     object 
 4   state                                                25 non-null     object 
 5   zip_code                                             25 non-null     object 
 6   latitude                                             25 non-null     float64
 7   longitude                                            25 non-null     float64
 8   avg_rating                                           25 non-null     float64


In [ ]:

df_rev_CANV_sindup=pd.read_parquet(r'archivos_ETL_intermedios\google_review_final.parquet' , engine='fastparquet')
ids_comunes = set(df_mt_pizza_CANV_sindup["gmap_id"]).intersection(set(df_rev_CA_sindup["gmap_id"]))

# Filtrar cada DataFrame
# Esto asegura: Que todos los negocios tengan al menos una review. + Que todas las reviews estén asociadas a un negocio existente.
df_google_piz = df_mt_pizza_CANV_sindup[df_mt_pizza_CANV_sindup["gmap_id"].isin(ids_comunes)]
df_google_piz.to_parquet(r'archivos_ETL_finales\GM.parquet' , engine='fastparquet')
df_google_rev_piz = df_rev_CANV_sindup[df_rev_CANV_sindup["gmap_id"].isin(ids_comunes)]
df_google_rev_piz.to_parquet(r'archivos_ETL_finales\GR.parquet' , engine='fastparquet')

In [ ]:
df_google_piz.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15 entries, 4 to 26
Data columns (total 46 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   gmap_id                                              15 non-null     object 
 1   business_name                                        15 non-null     object 
 2   address                                              15 non-null     object 
 3   city                                                 15 non-null     object 
 4   state                                                15 non-null     object 
 5   zip_code                                             15 non-null     object 
 6   latitude                                             15 non-null     float64
 7   longitude                                            15 non-null     float64
 8   avg_rating                                           15 non-null     float64


In [ ]:
# ---------------------------------------------------- YELP ---------------------------------------------------------------------
# Primero filtramos por ciudad
# leo el archivo de las pizzerias mt_pizza_NJNY para filtrar  las reviews  

yelp_pizza_CANV=pd.read_parquet(r'archivos_ETL_intermedios\yelp_pizza_CANV.parquet' , engine='fastparquet')
print(yelp_pizza_CANV["city"].head(10))
yelp_pizza_CANV = yelp_pizza_CANV[yelp_pizza_CANV['city'].isin(df_ciudades['city'])]



index
676     town and country
989                tampa
1717             lahaska
2170        philadelphia
2226            chalfont
2586            bellmawr
3000         new orleans
3090        philadelphia
3898        philadelphia
4050        philadelphia
Name: city, dtype: object


In [ ]:

filtered_reviews_YELP = df_reviews[df_reviews['business_id'].isin(yelp_pizza_CANV['business_id'])]
filtred_bussiness_YELP = yelp_pizza_CANV[yelp_pizza_CANV['business_id'].isin(filtered_reviews_YELP['business_id'])]
# guardo df_pizza_CANV_rev
filtered_reviews_YELP.to_parquet(r'archivos_ETL_finales\YR.parquet' , engine='fastparquet')
filtred_bussiness_YELP.to_parquet(r'archivos_ETL_finales\YB.parquet' , engine='fastparquet')

In [ ]:
yelp_pizza_CANV.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39 entries, 5397 to 147074
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   business_id   39 non-null     object 
 1   name          39 non-null     object 
 2   address       39 non-null     object 
 3   city          39 non-null     object 
 4   state         39 non-null     object 
 5   postal_code   39 non-null     object 
 6   latitude      39 non-null     float64
 7   longitude     39 non-null     float64
 8   stars         39 non-null     float64
 9   review_count  39 non-null     float64
dtypes: float64(4), object(6)
memory usage: 3.4+ KB


In [ ]:
# Filtrar ciudades presentes en Google
df_ciudades_filtro_g = df_ciudades[df_ciudades["city"].isin(df_google_piz["city"])].drop_duplicates(subset=["city"])

# Filtrar ciudades presentes en Yelp
df_ciudades_filtro_y = df_ciudades[df_ciudades["city"].isin(filtred_bussiness_YELP["city"])].drop_duplicates(subset=["city"])

# Concatenar ambas listas por filas
df_ciudades_filtro = pd.concat([df_ciudades_filtro_g, df_ciudades_filtro_y], ignore_index=True)

# Eliminar duplicados globales por nombre de ciudad
df_ciudades_filtro = df_ciudades_filtro.drop_duplicates(subset=["city"]).reset_index(drop=True)

# Guardar como archivo Parquet
df_ciudades_filtro.to_parquet(r'archivos_ETL_finales\ciudades.parquet', engine='fastparquet')


,city,Population,Estado
0,los angeles,3820914,Ca
2,san jose,969655,Ca
4,las vegas,660929,Nv
24,modesto,218915,Ca
44,sunnyvale,151967,Ca


In [ ]:
# -----------------------------
# 4. Configuración ARCHIVOS PARQUET 
# -----------------------------
# Variables de entorno, credenciales, rutas

In [ ]:
# Archivos finales en formato Parquet
dir_tabla_1=r'archivos_ETL_finales\GM.parquet'#Google Negocios

dir_tabla_2=r'archivos_ETL_finales\GR.parquet' # Google Reviews 

dir_tabla_3=r'archivos_ETL_finales\YB.parquet' # Yelp Negocios

dir_tabla_4=r'archivos_ETL_finales\YR.parquet' #Yelp Reviews



In [ ]:
# -----------------------------
# 5. Subida a GCS 
# -----------------------------

In [ ]:
from google.cloud import storage
import os

# Establecer la ruta al archivo de credenciales
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "credenciales.json"

# Parámetros
BUCKET_NAME = "proyecto_grupal_dw"
CARPETA_LOCAL = "archivos_ETL_finales"
CARPETA_DESTINO_GCS = "Dataset"
ARCHIVOS = ["GM.parquet", "GR.parquet", "YB.parquet", "YR.parquet","ciudades.parquet"]

# Crear cliente de GCS
storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)

# Subir archivos evitando sobrescritura
for archivo in ARCHIVOS:
    ruta_local = os.path.join(CARPETA_LOCAL, archivo)
    destino_gcs = f"{CARPETA_DESTINO_GCS}/{archivo}"
    blob = bucket.blob(destino_gcs)
    
   
    blob.upload_from_filename(ruta_local)
    print(f"✅ Subido: {archivo} a gs://{BUCKET_NAME}/{destino_gcs}")

✅ Subido: GM.parquet a gs://proyecto_grupal_dw/Dataset/GM.parquet
✅ Subido: GR.parquet a gs://proyecto_grupal_dw/Dataset/GR.parquet
✅ Subido: YB.parquet a gs://proyecto_grupal_dw/Dataset/YB.parquet
✅ Subido: YR.parquet a gs://proyecto_grupal_dw/Dataset/YR.parquet
✅ Subido: ciudades.parquet a gs://proyecto_grupal_dw/Dataset/ciudades.parquet


In [ ]:
from google.cloud import storage

client = storage.Client()
blobs = client.list_blobs("proyecto_grupal_dw", prefix="Dataset/")

for blob in blobs:
    print(blob.name)

Dataset/
Dataset/GM.parquet
Dataset/GR.parquet
Dataset/YB.parquet
Dataset/YR.parquet
Dataset/ciudades.parquet


In [ ]:
import os
from google.cloud import bigquery
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file("credenciales.json")
bq_client = bigquery.Client(credentials=credentials, project="helical-bongo-460417-n9")




# Verificación del archivo de credenciales
assert os.path.exists("credenciales.json"), "❌ Archivo de credenciales no encontrado"

# Configuración
PROJECT_ID = "helical-bongo-460417-n9"
DATASET_ID = "BQ_dataset"
BUCKET_NAME = "proyecto_grupal_dw"
CARPETA_DESTINO_GCS = "Dataset"
ARCHIVOS = ["GM.parquet", "GR.parquet", "YB.parquet", "YR.parquet","ciudades.parquet"]

# Carga a BigQuery
for archivo in ARCHIVOS:
    nombre_tabla = archivo.replace(".parquet", "")
    uri = f"gs://{BUCKET_NAME}/{CARPETA_DESTINO_GCS}/{archivo}"
    tabla_destino = f"{PROJECT_ID}.{DATASET_ID}.{nombre_tabla}"

    job_config = bigquery.LoadJobConfig(source_format=bigquery.SourceFormat.PARQUET)

    load_job = bq_client.load_table_from_uri(uri, tabla_destino, job_config=job_config)
    load_job.result()  # Espera a que termine

    print(f"✅ Tabla {nombre_tabla} cargada en BigQuery desde {uri}")

✅ Tabla GM cargada en BigQuery desde gs://proyecto_grupal_dw/Dataset/GM.parquet
✅ Tabla GR cargada en BigQuery desde gs://proyecto_grupal_dw/Dataset/GR.parquet
✅ Tabla YB cargada en BigQuery desde gs://proyecto_grupal_dw/Dataset/YB.parquet
✅ Tabla YR cargada en BigQuery desde gs://proyecto_grupal_dw/Dataset/YR.parquet
✅ Tabla ciudades cargada en BigQuery desde gs://proyecto_grupal_dw/Dataset/ciudades.parquet
